# FITS and RGB Image Viewer

Interactively view the original FITS files from each survey alongside the combined RGB image.

## Import Libraries

In [13]:
import os
from pathlib import Path
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, SymLogNorm
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output

## Define Paths and Helper Functions

In [14]:
# Define paths
RADCAT_F_PATH = "/home/abigaildeklerk/Desktop/DeKlerk_Models/Data/RADCAT_F"
RADCAT_F_UNPACKED = "/home/abigaildeklerk/Desktop/DeKlerk_Models/Data/RADCAT_F_UNPACKED"

SURVEYS = ['FIRST', 'LoTSS', 'NVSS']
SURVEY_COLORS = {'FIRST': 'Reds', 'LoTSS': 'Greens', 'NVSS': 'Blues'}

def load_fits_data(fits_path):
    """Load data from a FITS file."""
    try:
        with fits.open(fits_path) as hdul:
            data = hdul[0].data
            if data is None:
                for hdu in hdul[1:]:
                    if hdu.data is not None:
                        data = hdu.data
                        break
            
            if data is not None and data.ndim == 3:
                data = data[0]
            
            return data
    except Exception as e:
        return None

def load_rgb_image(png_path):
    """Load RGB image from PNG file."""
    try:
        img = Image.open(png_path).convert('RGB')
        return np.array(img, dtype=np.uint8)
    except Exception as e:
        return None

def get_source_ids():
    """Get all available source IDs from RADCAT_F_UNPACKED."""
    source_ids = set()
    
    # Get sources from RGB images
    for file in os.listdir(RADCAT_F_UNPACKED):
        if file.endswith('_RGB.png'):
            source_id = file.replace('_RGB.png', '')
            source_ids.add(source_id)
    
    return sorted(source_ids)

def get_fits_file_for_source(survey, source_id):
    """Get the FITS file path for a given survey and source ID."""
    survey_dir = os.path.join(RADCAT_F_PATH, survey)
    fits_file = os.path.join(survey_dir, f"{source_id}.fits")
    if os.path.exists(fits_file):
        return fits_file
    return None

def normalize_fits_data(data, percentile_range=(1, 99)):
    """Normalize FITS data for display."""
    data = np.nan_to_num(data, nan=0, posinf=0, neginf=0)
    if data.size == 0:
        return data
    
    v_min, v_max = np.percentile(data, percentile_range)
    if v_min == v_max:
        return np.zeros_like(data)
    
    return (data - v_min) / (v_max - v_min)

## Display Functions

In [15]:
def display_source_images(source_id):
    """Display the three FITS images and combined RGB image for a source."""
    
    # Create figure with 2x2 subplots
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    fig.suptitle(f'Source {source_id}: Survey Data + Combined RGB', fontsize=16, fontweight='bold')
    
    # Load and display FITS files from surveys
    for idx, survey in enumerate(SURVEYS):
        ax = axes[idx // 2, idx % 2]
        fits_path = get_fits_file_for_source(survey, source_id)
        
        if fits_path and os.path.exists(fits_path):
            fits_data = load_fits_data(fits_path)
            if fits_data is not None:
                normalized_data = normalize_fits_data(fits_data)
                im = ax.imshow(normalized_data, cmap=SURVEY_COLORS[survey], origin='lower')
                ax.set_title(f'{survey} Survey (FITS)', fontsize=12, fontweight='bold')
                plt.colorbar(im, ax=ax, label='Normalized Intensity')
            else:
                ax.text(0.5, 0.5, f'Could not load\n{survey} FITS file', 
                       ha='center', va='center', transform=ax.transAxes, fontsize=11)
                ax.set_title(f'{survey} Survey (FITS)', fontsize=12)
        else:
            ax.text(0.5, 0.5, f'{survey} file\nnot found', 
                   ha='center', va='center', transform=ax.transAxes, fontsize=11)
            ax.set_title(f'{survey} Survey (FITS)', fontsize=12)
        
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
    
    # Load and display combined RGB image
    ax = axes[1, 1]
    rgb_path = os.path.join(RADCAT_F_UNPACKED, f"{source_id}_RGB.png")
    
    if os.path.exists(rgb_path):
        rgb_data = load_rgb_image(rgb_path)
        if rgb_data is not None:
            ax.imshow(rgb_data, origin='lower')
            ax.set_title('Combined RGB\n(FIRST=R, LoTSS=G, NVSS=B)', fontsize=12, fontweight='bold')
        else:
            ax.text(0.5, 0.5, 'Could not load\nRGB image', 
                   ha='center', va='center', transform=ax.transAxes, fontsize=11)
            ax.set_title('Combined RGB', fontsize=12)
    else:
        ax.text(0.5, 0.5, 'RGB file\nnot found', 
               ha='center', va='center', transform=ax.transAxes, fontsize=11)
        ax.set_title('Combined RGB', fontsize=12)
    
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    
    plt.tight_layout()
    plt.show()

## Interactive Navigation

In [16]:
# Load available source IDs
source_ids = get_source_ids()
print(f"Found {len(source_ids)} sources")

if not source_ids:
    print("No sources found in RADCAT_F_UNPACKED!")
else:
    # State variable
    current_index = 0
    
    # Create widgets
    prev_button = widgets.Button(
        description='← Previous',
        button_style='info',
        tooltip='Show previous source'
    )
    
    next_button = widgets.Button(
        description='Next →',
        button_style='info',
        tooltip='Show next source'
    )
    
    source_selector = widgets.Dropdown(
        options=source_ids,
        value=source_ids[0],
        description='Source ID:'
    )
    
    status_label = widgets.HTML(
        value=f'<b>Viewing:</b> Source {source_ids[0]} (1 of {len(source_ids)})'
    )
    
    output_display = widgets.Output()
    
    def on_prev_clicked(b):
        """Handle previous button click."""
        global current_index
        current_index = (current_index - 1) % len(source_ids)
        source_selector.value = source_ids[current_index]
    
    def on_next_clicked(b):
        """Handle next button click."""
        global current_index
        current_index = (current_index + 1) % len(source_ids)
        source_selector.value = source_ids[current_index]
    
    def on_source_changed(change):
        """Handle source selection change."""
        global current_index
        current_index = source_ids.index(change['new'])
        update_display()
    
    def update_display():
        """Update the image display."""
        source_id = source_selector.value
        current_num = current_index + 1
        status_label.value = f'<b>Viewing:</b> Source {source_id} ({current_num} of {len(source_ids)})'
        
        with output_display:
            clear_output(wait=True)
            display_source_images(source_id)
    
    # Attach event handlers
    prev_button.on_click(on_prev_clicked)
    next_button.on_click(on_next_clicked)
    source_selector.observe(on_source_changed, names='value')
    
    # Layout
    button_layout = widgets.HBox([prev_button, next_button], 
                                 layout=widgets.Layout(justify_content='center', margin='10px'))
    selector_layout = widgets.HBox([source_selector], 
                                   layout=widgets.Layout(justify_content='center', margin='10px'))
    
    # Display controls and initial image
    print("\n" + "="*80)
    print("FITS and RGB Image Viewer")
    print("="*80)
    display(selector_layout)
    display(button_layout)
    display(status_label)
    display(output_display)
    
    # Show initial image
    update_display()

Found 2510 sources

FITS and RGB Image Viewer


HTML(value='<b>Viewing:</b> Source 10000 (1 of 2510)')

Output()